Importación de módulos necesarios

In [ ]:
import pandas as pd

Carga de conjuntos de datos a utilizarse

In [ ]:
ventas = pd.read_csv('../data/ventas.csv')

In [ ]:
temperaturas = pd.read_csv('../data/temperaturas.csv')

In [ ]:
precipitaciones = pd.read_csv('../data/precipitaciones.csv',sep=';')

Contenido de `ventas`

In [ ]:
ventas

##### Observaremos el período de ventas registrado

In [ ]:
print("Período: Desde",ventas.fecha_vta.min(),"hasta",ventas.fecha_vta.max())

##### Observaremos los locales de venta registrados

In [ ]:
print("Locales: ",ventas.local.unique())
print("Se aclara que ambos locales están en el departamento Central")

Con lo observado ya se sabe el período y el lugar de donde se debe extraer los datos de meteorología

Contenido de `temperaturas`

In [ ]:
temperaturas

##### Extraemos los datos dentro del período determinado y renombramos las columnas

In [ ]:
temperaturas.columns = ['fecha','minima','maxima','media']
temperaturas = temperaturas.query('fecha >= "2021-01-02" and fecha <= "2021-09-30"')
temperaturas

Contenido de `precipitaciones`

In [ ]:
precipitaciones

##### Extraemos los datos dentro del período determinado, eliminamos la fila *TOTAL* (no es necesario) y renombramos las columnas

In [ ]:
precipitaciones.columns = ['dia','ene','feb','mar','abr','may','jun','jul','ago','sep','oct','nov','dic']
precipitaciones.drop(labels=[0],axis=0,inplace=True)
precipitaciones.drop(labels=['dia','oct','nov','dic'],axis=1,inplace=True)
precipitaciones

Reemplazamos comas por puntos, a fin que pandas interprete los números como valores flotantes

In [ ]:
precipitaciones = precipitaciones.replace(',', '.', regex=True).astype(float)

Verificación de datos faltantes

`Ventas`

In [ ]:
faltantes = ventas.isnull().sum()
print("Cantidad de valores faltantes en fecha_vta:",faltantes.fecha_vta)
print("Cantidad de valores faltantes en local:",faltantes.local)
print("Cantidad de valores faltantes en secciones:",faltantes.desc_secciones)
print("Cantidad de valores faltantes en sub_secciones:",faltantes.desc_sub_secciones)
print("Cantidad de valores faltantes en cantidad:",faltantes.cantidad)

El análisis no se podrá realizar correctamente con las secciones y subsecciones faltantes, se eliminan esas filas

In [ ]:
ventas.dropna(subset=['desc_secciones', 'desc_sub_secciones'],inplace=True)

`Temperaturas`

In [ ]:
faltantes = temperaturas.isnull().sum()
print("Cantidad de valores faltantes en fecha:",faltantes.fecha)
print("Cantidad de valores faltantes en minima:",faltantes.minima)
print("Cantidad de valores faltantes en maxima:",faltantes.maxima)
print("Cantidad de valores faltantes en media:",faltantes.media)

`Precipitaciones`

In [ ]:
faltantes = precipitaciones.isnull().sum()
print("Cantidad de valores faltantes en ene:",faltantes.ene)
print("Cantidad de valores faltantes en feb:",faltantes.feb)
print("Cantidad de valores faltantes en mar:",faltantes.mar)
print("Cantidad de valores faltantes en abr:",faltantes.abr)
print("Cantidad de valores faltantes en may:",faltantes.may)
print("Cantidad de valores faltantes en jun:",faltantes.jun)
print("Cantidad de valores faltantes en jul:",faltantes.jul)
print("Cantidad de valores faltantes en ago:",faltantes.ago)
print("Cantidad de valores faltantes en sep:",faltantes.sep)
print()
print("Los valores faltantes que se muestra corresponden a fechas que no existen en el calendario, lo cual tiene sentido")

No se realizará verificación de duplicados, en los tres conjuntos de datos es perfectamente posible que se repitan los eventos (mismas ventas en el día, mismas temperaturas y precipitaciones)

Verificación de valores atípicos

`Ventas`

In [ ]:
print("Los distintos valores de secciones:",ventas.desc_secciones.unique())
ventas.query("desc_secciones=='Servicios '")

Se observa que la sección *Servicios* no son ventas reales, por lo que se eliminan esas filas

In [ ]:
ventas.drop(ventas.query("desc_secciones=='Servicios '").index,inplace=True)
# Proseguimos
print("El valor mínimo de cantidad:",ventas.cantidad.min())
print("El valor máximo de cantidad:",ventas.cantidad.max())

Se detectan cantidades negativas, se eliminan esas filas

In [ ]:
ventas.drop(ventas.query("cantidad<=0").index,inplace=True)

`Temperaturas`

In [ ]:
print("El valor mínimo de la temperatura mínima:",temperaturas.minima.min())
print("El valor máximo de la temperatura mínima:",temperaturas.minima.max())
print("El valor mínimo de la temperatura máxima:",temperaturas.maxima.min())
print("El valor máximo de la temperatura máxima:",temperaturas.maxima.max())
print("El valor mínimo de la temperatura media:",temperaturas.media.min())
print("El valor máximo de la temperatura media:",temperaturas.media.max())

`Precipitaciones`

In [ ]:
print("El valor mínimo de precipitaciones:",precipitaciones.min().min())

Normalización de formatos de fecha

In [ ]:
# Convertir la columna 'fecha_vta' en ventas a tipo datetime
ventas['fecha_vta'] = pd.to_datetime(ventas['fecha_vta'])

# Convertir la columna 'fecha' en temperaturas a tipo datetime
temperaturas['fecha'] = pd.to_datetime(temperaturas['fecha'])

print("Tipos de datos después de la conversión:")
print("Ventas - fecha_vta:", ventas['fecha_vta'].dtype)
print("Temperaturas - fecha:", temperaturas['fecha'].dtype)

La tabla de precipitaciones está en formato ancho (un mes por columna). Para poder unirla con las otras tablas, necesitamos que tenga una columna de `fecha` y una columna con el `valor` de precipitación para ese día.

In [ ]:
# Resetear el índice para que 'dia' sea una columna regular
precipitaciones_long = precipitaciones.reset_index()
precipitaciones_long = precipitaciones_long.rename(columns={'index': 'dia'})

# Melt para transformar de formato ancho a largo
# Los 'id_vars' son las columnas que queremos mantener como identificadores (en este caso, 'dia')
# Los 'var_name' será la nueva columna que contendrá los nombres de los meses
# Los 'value_name' será la nueva columna que contendrá los valores de precipitación
precipitaciones_long = precipitaciones_long.melt(id_vars=['dia'], var_name='mes', value_name='precipitacion')

# Mapear nombres de meses a números de meses
month_map = {
    'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'ago': 8, 'sep': 9
}
precipitaciones_long['mes_num'] = precipitaciones_long['mes'].map(month_map)

# Añadir un año (basado en el rango de fechas de ventas y temperaturas)
# Asumimos que estos datos son para 2021
precipitaciones_long['anio'] = 2021

# Crear la columna de fecha completa
# Asegurarse de manejar los días 31 para meses que no los tienen
precipitaciones_long['fecha_str'] = precipitaciones_long['anio'].astype(str) + '-' + \
                                  precipitaciones_long['mes_num'].astype(str) + '-' + \
                                  precipitaciones_long['dia'].astype(str)

# Convertir a datetime, forzando errores para fechas inválidas (e.g., 31 de febrero)
precipitaciones_long['fecha'] = pd.to_datetime(precipitaciones_long['fecha_str'], errors='coerce')

# Eliminar filas con fechas inválidas que resultaron en NaT
precipitaciones_long.dropna(subset=['fecha'], inplace=True)

# Seleccionar solo las columnas relevantes y ordenar
precipitaciones_long = precipitaciones_long[['fecha', 'precipitacion']].sort_values(by='fecha').reset_index(drop=True)

precipitaciones_long

Ahora que todas las tablas tienen una columna de fecha en el formato correcto, podemos unirlas. Comenzaremos uniendo ventas con temperaturas, y luego el resultado con precipitaciones.

In [ ]:
# Unir ventas con temperaturas
# Usamos un 'left merge' para mantener todas las ventas y añadir la información de temperatura cuando coincida la fecha.
# Renombramos 'fecha_vta' en ventas a 'fecha' para que coincida con 'temperaturas'
ventas_temp = pd.merge(ventas.rename(columns={'fecha_vta': 'fecha'}),
                       temperaturas,
                       on='fecha',
                       how='left')

# Unir el resultado con precipitaciones
# Nuevamente, usamos un 'left merge'.
df_final = pd.merge(ventas_temp,
                    precipitaciones_long,
                    on='fecha',
                    how='left')

Diccionario de datos final


In [ ]:
df_final


In [ ]:
df_final.to_csv('../output/df_final.csv', index=False)